# 02 — Cash Flow Forecasting with TimesFM

Generate cash flow forecasts using BigQuery's AI.FORECAST function powered by the TimesFM foundation model. No model training required.

In [ ]:
import os
from google.cloud import bigquery

PROJECT_ID = os.environ.get('PROJECT_ID', 'your-project-id')
DATASET_ID = 'cash_agent_demo'
client = bigquery.Client(project=PROJECT_ID)

## Step 1: Explore Historical Data

Daily net cash flow by currency from the cash journal.

In [ ]:
query = f"""
SELECT posting_date, currency,
       SUM(CASE WHEN transaction_type='INFLOW' THEN amount ELSE -amount END) AS net_cash_flow
FROM `{PROJECT_ID}.{DATASET_ID}.cash_journal`
GROUP BY posting_date, currency
ORDER BY currency, posting_date
"""
df = client.query(query).to_dataframe()
print(f"Training data: {len(df)} rows")
print(f"Date range: {df.posting_date.min()} to {df.posting_date.max()}")
print(f"Currencies: {df.currency.unique().tolist()}")
df.groupby('currency')['net_cash_flow'].describe()

## Step 2: Generate Forecast with AI.FORECAST (TimesFM)

AI.FORECAST uses the TimesFM foundation model — no model creation or training needed. It takes historical data directly and produces forecasts.

In [ ]:
forecast_sql = f"""
SELECT
    forecast_timestamp AS forecast_date,
    forecast_value AS net_cash_flow,
    prediction_interval_lower_bound AS lower_bound,
    prediction_interval_upper_bound AS upper_bound,
    currency
FROM AI.FORECAST(
    (SELECT posting_date, currency,
            SUM(CASE WHEN transaction_type='INFLOW' THEN amount ELSE -amount END) AS net_cash_flow
     FROM `{PROJECT_ID}.{DATASET_ID}.cash_journal`
     GROUP BY posting_date, currency),
    data_col => 'net_cash_flow',
    timestamp_col => 'posting_date',
    id_cols => ['currency'],
    horizon => 90,
    confidence_level => 0.95
)
ORDER BY currency, forecast_timestamp
"""

print("Running AI.FORECAST with TimesFM...")
forecast_df = client.query(forecast_sql).to_dataframe()
print(f"Forecast rows: {len(forecast_df)}")
forecast_df.head(10)

## Step 3: Detect Anomalies with AI.DETECT_ANOMALIES (TimesFM)

Use the same TimesFM model to detect anomalous cash flow patterns in recent data.

In [ ]:
anomaly_sql = f"""
SELECT *
FROM AI.DETECT_ANOMALIES(
    (SELECT posting_date, currency,
            SUM(CASE WHEN transaction_type='INFLOW' THEN amount ELSE -amount END) AS net_cash_flow
     FROM `{PROJECT_ID}.{DATASET_ID}.cash_journal`
     WHERE posting_date < DATE_SUB(CURRENT_DATE(), INTERVAL 14 DAY)
     GROUP BY posting_date, currency),
    (SELECT posting_date, currency,
            SUM(CASE WHEN transaction_type='INFLOW' THEN amount ELSE -amount END) AS net_cash_flow
     FROM `{PROJECT_ID}.{DATASET_ID}.cash_journal`
     WHERE posting_date >= DATE_SUB(CURRENT_DATE(), INTERVAL 14 DAY)
     GROUP BY posting_date, currency),
    data_col => 'net_cash_flow',
    timestamp_col => 'posting_date',
    id_cols => ['currency'],
    anomaly_prob_threshold => 0.95
)
ORDER BY anomaly_probability DESC
"""

anomaly_df = client.query(anomaly_sql).to_dataframe()
print(f"Total data points checked: {len(anomaly_df)}")
print(f"Anomalies detected: {anomaly_df['is_anomaly'].sum()}")
anomaly_df[anomaly_df['is_anomaly'] == True]

## Step 4: Visualize 90-Day Forecast

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
for i, ccy in enumerate(['USD', 'EUR', 'GBP']):
    ccy_df = forecast_df[forecast_df.currency == ccy]
    ax = axes[i]
    ax.plot(ccy_df.forecast_date, ccy_df.net_cash_flow, label='Forecast', color='#0070F2')
    ax.fill_between(ccy_df.forecast_date, ccy_df.lower_bound, ccy_df.upper_bound,
                    alpha=0.2, color='#0070F2', label='95% CI')
    ax.set_title(f'{ccy} Daily Net Cash Flow Forecast')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()